# Frameworks 00 - LangGraph

Objetivo: ejecutar el adaptador LangGraph real sobre el Provider seleccionado
por `.env`, producir una respuesta humana con evidencia de Tool y conservar un
StateGraph condicional observable. El control offline permanece determinista.

**Lugar en el modelo:** LangGraph controla el grafo/loop; el Provider controla
donde ocurre la inferencia. Son decisiones independientes.

**Evidencia exigida:** `human_result` debe registrar Provider, Framework,
modelo, Tool, respuesta publica y linaje; el SDK real tambien debe compilar y
ejecutar rutas sync/async y un grafo condicional.

**Límite de la evidencia:** `python-runtime` solo certifica la integracion
offline del SDK. Una prueba live debe observar un Provider LM real y no puede
presentar JSON tecnico como respuesta humana.

## Parametros de la demostracion

`.env` es la fuente canonica. `AGENTIC_SYSTEMS_PROVIDER` puede fijar el Provider
o dejar `auto`; el flag live del Provider seleccionado autoriza la llamada.
`RUN_LANGGRAPH_LIVE` es solamente un override opcional.

| Variable | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_PROVIDER | auto | Provider utilizado por LangGraph. |
| RUN_LANGGRAPH_LIVE | flag del Provider | Override opcional. |
| Framework | langgraph | Compila y ejecuta el SDK real. |

## 1) Provider y Framework independientes


In [ ]:
import os

import agentic_systems as toolkit


def _enabled(value: str | None) -> bool:
    return str(value or "").strip().lower() in {"1", "true", "yes"}


requested_provider = os.getenv("AGENTIC_SYSTEMS_PROVIDER", "auto").strip() or "auto"
candidate_runtime = toolkit.runtime(provider=requested_provider)
candidate_description = candidate_runtime.describe()
selected_provider = candidate_description["selected_provider"]
provider_live_flag = (
    f"RUN_{selected_provider.removesuffix('-runtime').replace('-', '_').upper()}_LIVE"
    if selected_provider and selected_provider != "python-runtime"
    else None
)
provider_live_value = os.getenv(provider_live_flag, "0") if provider_live_flag else "0"
RUN_LIVE = (
    selected_provider != "python-runtime"
    and _enabled(os.getenv("RUN_LANGGRAPH_LIVE", provider_live_value))
)
runtime = candidate_runtime if RUN_LIVE else toolkit.runtime(provider="python-runtime")
runtime_description = runtime.describe()
execution_kind = "live-language-model" if RUN_LIVE else "offline-deterministic-control"

framework = toolkit.framework("langgraph")
profile = toolkit.integrations.framework_profile("langgraph")
toolkit.show_json(
    {
        "execution_kind": execution_kind,
        "requested_provider": requested_provider,
        "provider_live_flag": provider_live_flag,
        "provider_live_authorized": RUN_LIVE,
        "runtime": runtime_description,
        "framework": framework.inspect(),
        "profile": profile.to_dict(),
    },
    title="Provider x LangGraph preflight",
)


## 2) Agent ejecutado dentro de un grafo real de un nodo


In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}


agent = toolkit.agent(
    name="langgraph_inspector",
    instructions=(
        "Usa inspect_public_api para verificar el simbolo solicitado. "
        "Despues responde una sola frase natural en espanol; no devuelvas JSON."
    ),
    runtime=runtime,
    tools=[inspect_public_api],
    framework=framework,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_turns=4, max_tool_calls=1, repair=True, max_repairs=2),
)
agent.prepare()
sync_request = (
    "Verifica si graph pertenece a la API publica instalada."
    if RUN_LIVE
    else {"tool": "inspect_public_api", "input": {"symbol": "graph"}}
)
async_request = (
    "Verifica si framework pertenece a la API publica instalada."
    if RUN_LIVE
    else {"tool": "inspect_public_api", "input": {"symbol": "framework"}}
)
sync_result = agent.run(sync_request, mode="eval")
async_result = await agent.arun(async_request, mode="eval")
toolkit.show_json(
    {
        "sync": {
            "ok": sync_result.ok,
            "errors": sync_result.errors,
            "tools": [event.name for event in sync_result.tool_events],
            "contract_repairs": sync_result.meta.get("contract_repairs", 0),
        },
        "async": {
            "ok": async_result.ok,
            "errors": async_result.errors,
            "tools": [event.name for event in async_result.tool_events],
            "contract_repairs": async_result.meta.get("contract_repairs", 0),
        },
    },
    title="LangGraph sync/async contract gate",
)
assert sync_result.ok and async_result.ok
assert sync_result.engine == async_result.engine == runtime_description["selected_provider"]
assert sync_result.meta["framework_adapter"] == "langgraph"
assert async_result.meta["framework_adapter"] == "langgraph"
for observed in (sync_result, async_result):
    observed.raise_if_inconsistent()
    assert not observed.meta.get("fallback_provider")
    assert any(event.name == "inspect_public_api" and event.ok for event in observed.tool_events)
if RUN_LIVE:
    assert sync_result.engine != "python-runtime"
    assert sync_result.text and not sync_result.text.lstrip().startswith("{"), sync_result.text
    assert async_result.text and not async_result.text.lstrip().startswith("{"), async_result.text

toolkit.human_result(
    sync_result,
    title="LangGraph live RunResult" if RUN_LIVE else "LangGraph offline deterministic control",
    show_lineage=True,
)
toolkit.show_json(
    {
        "execution_kind": execution_kind,
        "native_agent": type(agent.native_agent).__name__,
        "native_result": type(sync_result.native_result).__name__,
        "sync_framework": sync_result.meta["framework_adapter"],
        "async_framework": async_result.meta["framework_adapter"],
    },
    title="Real one-node StateGraph",
)


## 3) Estado, routing y ramas con toolkit.graph


In [ ]:
def classify(state: dict) -> dict:
    return {**state, "route": "accepted" if state["score"] >= 0.5 else "rejected"}

def accept(state: dict) -> dict:
    return {**state, "decision": "accepted"}

def reject(state: dict) -> dict:
    return {**state, "decision": "rejected"}

def route(state: dict) -> str:
    return state["route"]

app = toolkit.graph(
    name="langgraph_routing",
    engine="langgraph",
    state=dict,
    nodes={"classify": classify, "accept": accept, "reject": reject},
    edges=[("START", "classify"), ("accept", "END"), ("reject", "END")],
    conditional_edges=[
        ("classify", route, {"accepted": "accept", "rejected": "reject"}),
    ],
)
accepted_state = app.run({"score": 0.9})
rejected_state = await app.arun({"score": 0.1})
assert accepted_state["decision"] == "accepted"
assert rejected_state["decision"] == "rejected"
toolkit.show_json(
    {
        "engine": app.engine,
        "framework": app.framework,
        "native_graph": type(app.native).__name__,
        "accepted": accepted_state,
        "rejected": rejected_state,
    },
    title="Conditional StateGraph",
)


## 4) Lineage desde el estado ejecutado


In [ ]:
lineage = app.lineage(
    accepted_state,
    question="Que rama eligio LangGraph?",
    goal="Conservar estado, routing y decision.",
    answer_keys=("decision", "route"),
)
toolkit.show(lineage, title="LangGraph lineage")

api_coverage = [
    "toolkit.runtime", "toolkit.framework", "toolkit.integrations.framework_profile",
    "toolkit.tool", "toolkit.agent", "Agent.prepare", "Agent.native_agent",
    "agent.run", "agent.arun", "RunResult.native_result", "toolkit.RunPolicy", "toolkit.AgentContract",
    "toolkit.graph", "GraphApp.run", "GraphApp.arun", "GraphApp.lineage",
    "toolkit.human_result", "toolkit.show", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="LangGraph API coverage")


## Resultado e interpretacion

Cuando el preflight declara `live-language-model`, el Provider seleccionado por
`.env` realiza inferencia y LangGraph controla el grafo del Agent. La respuesta
publica debe ser humana y su linaje debe contener la Tool observada.

El StateGraph condicional demuestra routing nativo por separado. Cuando el
preflight declara `offline-deterministic-control`, `python-runtime` solo verifica
la integracion del SDK y no se presenta como modelo de lenguaje.